In [ ]:
# !pip install python-docx

In [ ]:
# import re
# import json
# from docx import Document

# def normalize_vietnamese_text(text: str) -> str:
#     """Chuẩn hóa khoảng trắng và các ký tự xuống dòng thừa trong văn bản."""
#     text = re.sub(r'[\r\n]+', ' ', text)
#     text = re.sub(r'\s+', ' ', text)
#     return text.strip()

# def split_law_hierarchy_from_docx(docx_path: str, source_title: str) -> list:
#     """
#     Đọc file Word (.docx) chứa văn bản Nghị định/Luật và tách thành các chunk cấu trúc.
    
#     :param docx_path: Đường dẫn tới file Word (.docx)
#     :param source_title: Tên đầy đủ của văn bản nguồn (VD: 'TOÀN VĂN: Nghị định 168/2024/NĐ-CP...')
#     :return: Danh sách các dictionary chứa chunk luật chuẩn hóa.
#     """
#     doc = Document(docx_path)
#     processed_chunks = []
    
#     # 1. Trích xuất Prefix định danh văn bản (VD: "ND168" từ "Nghị định 168/2024/NĐ-CP")
#     law_id_match = re.search(r'định\s+(?:số\s+)?(\d+)', source_title, re.IGNORECASE)
#     law_prefix = f"ND{law_id_match.group(1)}" if law_id_match else "LAW"
    
#     # 2. Regex nâng cấp để nhận diện chính xác Điều, Khoản, Điểm
#     article_regex = re.compile(r'^(Điều\s*\d+)[\.\-:\s]\s*(.*)', re.IGNORECASE)
#     clause_regex = re.compile(r'^(\d+)[\.\-:\s]\s*(.*)')
#     point_regex = re.compile(r'^([a-zđA-ZĐ])[\)\.\-]\s*(.*)', re.IGNORECASE)

#     current_article_num = ""
#     current_article_title = ""
#     current_clause_num = ""
#     clause_prefix_text = ""

#     # Duyệt qua từng đoạn văn (paragraph) trong file Word
#     for paragraph in doc.paragraphs:
#         line = paragraph.text.strip()
#         if not line:
#             continue
            
#         # -----------------------------------------------------------------
#         # 1. NHẬN DIỆN ĐIỀU
#         # -----------------------------------------------------------------
#         article_match = article_regex.match(line)
#         if article_match:
#             current_article_num = article_match.group(1).replace(" ", "")  # VD: "Điều1"
#             current_article_title = article_match.group(2).strip()        # VD: "Phạm vi điều chỉnh"
#             # Reset trạng thái khoản và điểm khi sang Điều mới
#             current_clause_num = ""
#             clause_prefix_text = ""
#             continue
            
#         if not current_article_num:
#             continue
            
#         # -----------------------------------------------------------------
#         # 2. NHẬN DIỆN KHOẢN
#         # -----------------------------------------------------------------
#         clause_match = clause_regex.match(line)
#         if clause_match:
#             current_clause_num = clause_match.group(1)                   # VD: "1"
#             clause_prefix_text = clause_match.group(2).strip()           # VD: "Nghị định này quy định về:"
            
#             chunk_id = f"{law_prefix}_{current_article_num}_K{current_clause_num}"
#             full_context = f"[{source_title} - {current_article_num}. {current_article_title}] {line}"
            
#             processed_chunks.append({
#                 "id": chunk_id,
#                 "level": "Clause",
#                 "source": source_title,
#                 "article": current_article_title,
#                 "clause": current_clause_num,
#                 "point": None,
#                 "raw_text": line,
#                 "full_legal_text": normalize_vietnamese_text(full_context)
#             })
#             continue
            
#         # -----------------------------------------------------------------
#         # 3. NHẬN DIỆN ĐIỂM
#         # -----------------------------------------------------------------
#         point_match = point_regex.match(line)
#         if point_match:
#             current_point_letter = point_match.group(1).lower()          # VD: "a"
#             point_text = point_match.group(2).strip()
            
#             # ID định dạng chuẩn: ND168_Điều1_K1_Da
#             chunk_id = f"{law_prefix}_{current_article_num}_K{current_clause_num}_D{current_point_letter}"
            
#             # Ghép ngữ cảnh Khoản nếu có
#             clause_context = f"Theo Khoản {current_clause_num}: {clause_prefix_text} " if current_clause_num else ""
            
#             full_context = (
#                 f"[{source_title} - {current_article_num}. {current_article_title}] "
#                 f"{clause_context}"
#                 f"Điểm {current_point_letter}) {point_text}"
#             )
            
#             processed_chunks.append({
#                 "id": chunk_id,
#                 "level": "Point",
#                 "source": source_title,
#                 "article": current_article_title,
#                 "clause": current_clause_num if current_clause_num else None,
#                 "point": current_point_letter,
#                 "raw_text": line,
#                 "full_legal_text": normalize_vietnamese_text(full_context)
#             })
#             continue

#     return processed_chunks


# # =========================================================================
# # VÍ DỤ SỬ DỤNG VÀ XUẤT RA FILE JSON
# # =========================================================================
# if __name__ == "__main__":
#     # Đường dẫn tới file docx Nghị định 168 của bạn
#     DOCX_FILE_PATH = "168.2024.NĐ.CP.doc"
#     SOURCE_TITLE = "TOÀN VĂN: Nghị định 168/2024/NĐ-CP quy định xử phạt vi phạm hành chính về trật tự, an toàn giao thông đường bộ"

#     try:
#         # Chạy hàm xử lý
#         results = split_law_hierarchy_from_docx(DOCX_FILE_PATH, SOURCE_TITLE)
        
#         # In thử 2 kết quả đầu tiên
#         print(json.dumps(results[:2], ensure_ascii=False, indent=4))
        
#         # Lưu ra file json để làm Dataset cho RAG / Graph DB
#         with open("nghi_dinh_168_parsed.json", "w", encoding="utf-8") as f:
#             json.dump(results, f, ensure_ascii=False, indent=4)
            
#         print(f"\n[Thành công] Đã trích xuất {len(results)} chunks dữ liệu!")
        
#     except FileNotFoundError:
#         print(f"Lỗi: Không tìm thấy file tại đường dẫn '{DOCX_FILE_PATH}'. Vui lòng kiểm tra lại đường dẫn file .docx.")

In [7]:
import os
import re
import json

def normalize_vietnamese_text(text: str) -> str:
    """Chuẩn hóa khoảng trắng và các ký tự xuống dòng, ký tự điều khiển thừa."""
    if not text:
        return ""
    text = re.sub(r'[\r\n\x07\x0c]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_paragraphs_from_file(file_path: str) -> list:
    """
    Tự động nhận diện định dạng (.doc hoặc .docx) để trích xuất các đoạn văn bản.
    """
    abs_path = os.path.abspath(file_path)
    if not os.path.exists(abs_path):
        raise FileNotFoundError(f"Không tìm thấy file tại đường dẫn: {abs_path}")

    ext = os.path.splitext(abs_path)[1].lower()

    if ext == '.docx':
        from docx import Document
        doc = Document(abs_path)
        return [p.text for p in doc.paragraphs if p.text.strip()]

    elif ext == '.doc':
        try:
            import win32com.client as win32
        except ImportError:
            raise ImportError("Để đọc file .doc trực tiếp, vui lòng cài đặt pywin32: pip install pywin32")
        
        print("Đang kết nối MS Word ngầm để đọc file .doc...")
        word = win32.Dispatch("Word.Application")
        word.Visible = False
        try:
            doc = word.Documents.Open(abs_path)
            paragraphs = [p.Range.Text for p in doc.Paragraphs if p.Range.Text.strip()]
            doc.Close(False)
            return paragraphs
        finally:
            word.Quit()
    else:
        raise ValueError(f"Định dạng file '{ext}' không hỗ trợ! Chỉ chấp nhận .doc hoặc .docx.")

def parse_and_save_law_document(
    input_file_path: str, 
    source_title: str, 
    output_json_path: str = "nghi_dinh_168_parsed.json"
) -> list:
    """
    Bóc tách cấu trúc Nghị định và lưu trực tiếp ra file JSON.
    """
    raw_paragraphs = extract_paragraphs_from_file(input_file_path)
    
    # Trích xuất prefix luật (VD: "ND168" từ tên văn bản)
    law_id_match = re.search(r'định\s+(?:số\s+)?(\d+)', source_title, re.IGNORECASE)
    law_prefix = f"ND{law_id_match.group(1)}" if law_id_match else "LAW"

    # Regex nhận diện cấu trúc
    article_regex = re.compile(r'^(Điều\s*\d+)[\.\-:\s]\s*(.*)', re.IGNORECASE)
    clause_regex = re.compile(r'^(\d+)[\.\-:\s]\s*(.*)')
    point_regex = re.compile(r'^([a-zđA-ZĐ])[\)\.\-]\s*(.*)', re.IGNORECASE)

    processed_chunks = []
    current_article_num = ""
    current_article_title = ""
    current_clause_num = ""
    clause_prefix_text = ""
    
    # Biến lưu trữ chunk hiện tại để nối dòng nếu có văn bản kéo dài (gạch đầu dòng, ví dụ,...)
    current_chunk = None

    for p in raw_paragraphs:
        line = normalize_vietnamese_text(p)
        if not line:
            continue

        # -----------------------------------------------------------------
        # 1. NHẬN DIỆN ĐIỀU
        # -----------------------------------------------------------------
        article_match = article_regex.match(line)
        if article_match:
            current_article_num = article_match.group(1).replace(" ", "")  # VD: "Điều1"
            current_article_title = article_match.group(2).strip()        # VD: "Phạm vi điều chỉnh"
            current_clause_num = ""
            clause_prefix_text = ""
            current_chunk = None
            continue

        if not current_article_num:
            continue

        # -----------------------------------------------------------------
        # 2. NHẬN DIỆN KHOẢN
        # -----------------------------------------------------------------
        clause_match = clause_regex.match(line)
        if clause_match:
            current_clause_num = clause_match.group(1)
            clause_prefix_text = clause_match.group(2).strip()
            
            chunk_id = f"{law_prefix}_{current_article_num}_K{current_clause_num}"
            full_context = f"[{source_title} - {current_article_num}. {current_article_title}] {line}"
            
            current_chunk = {
                "id": chunk_id,
                "level": "Clause",
                "source": source_title,
                "article": current_article_title,
                "clause": current_clause_num,
                "point": None,
                "raw_text": line,
                "full_legal_text": normalize_vietnamese_text(full_context)
            }
            processed_chunks.append(current_chunk)
            continue

        # -----------------------------------------------------------------
        # 3. NHẬN DIỆN ĐIỂM
        # -----------------------------------------------------------------
        point_match = point_regex.match(line)
        if point_match:
            current_point_letter = point_match.group(1).lower()
            point_text = point_match.group(2).strip()
            
            # Khớp ID dạng chuẩn: ND168_Điều1_K1_Da
            chunk_id = f"{law_prefix}_{current_article_num}_K{current_clause_num}_D{current_point_letter}"
            
            clause_context = f"Theo Khoản {current_clause_num}: {clause_prefix_text} " if current_clause_num else ""
            full_context = (
                f"[{source_title} - {current_article_num}. {current_article_title}] "
                f"{clause_context}"
                f"Điểm {current_point_letter}) {point_text}"
            )
            
            current_chunk = {
                "id": chunk_id,
                "level": "Point",
                "source": source_title,
                "article": current_article_title,
                "clause": current_clause_num if current_clause_num else None,
                "point": current_point_letter,
                "raw_text": line,
                "full_legal_text": normalize_vietnamese_text(full_context)
            }
            processed_chunks.append(current_chunk)
            continue

        # -----------------------------------------------------------------
        # 4. XỬ LÝ DÒNG NỐI TIẾP (Gạch đầu dòng, đoạn văn kéo dài thuộc Điểm/Khoản)
        # -----------------------------------------------------------------
        if current_chunk is not None:
            # Nối văn bản dòng mới vào raw_text
            current_chunk["raw_text"] += f" {line}"
            
            # Cập nhật lại full_legal_text chuẩn chỉnh
            if current_chunk["level"] == "Clause":
                full_context = f"[{source_title} - {current_article_num}. {current_article_title}] {current_chunk['raw_text']}"
            else: # Point
                clause_context = f"Theo Khoản {current_clause_num}: {clause_prefix_text} " if current_clause_num else ""
                full_context = (
                    f"[{source_title} - {current_article_num}. {current_article_title}] "
                    f"{clause_context}"
                    f"{current_chunk['raw_text']}"
                )
            current_chunk["full_legal_text"] = normalize_vietnamese_text(full_context)

    # -----------------------------------------------------------------
    # 5. GHI DỮ LIỆU RA FILE JSON
    # -----------------------------------------------------------------
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(processed_chunks, f, ensure_ascii=False, indent=4)
        
    print(f"\n XỬ LÝ THÀNH CÔNG!")
    print(f"Tổng số chunks đã trích xuất: {len(processed_chunks)}")
    print(f"File JSON đã được lưu tại: {os.path.abspath(output_json_path)}")
    
    return processed_chunks


# =========================================================================
# CHẠY CHƯƠNG TRÌNH
# =========================================================================
if __name__ == "__main__":
    # Thay đổi đường dẫn file input của bạn (hỗ trợ cả .doc và .docx)
    INPUT_FILE = "./168.2024.NĐ.CP.doc" 
    
    SOURCE_TITLE = "TOÀN VĂN: Nghị định 168/2024/NĐ-CP quy định xử phạt vi phạm hành chính về trật tự, an toàn giao thông đường bộ"
    OUTPUT_JSON = "nghi_dinh_168_parsed.json"

    # Chạy hàm bóc tách và xuất JSON
    chunks = parse_and_save_law_document(INPUT_FILE, SOURCE_TITLE, OUTPUT_JSON)

Đang kết nối MS Word ngầm để đọc file .doc...

 XỬ LÝ THÀNH CÔNG!
Tổng số chunks đã trích xuất: 1242
File JSON đã được lưu tại: d:\UIT\sdhuit\course\term3\TriThuc\final\temp1\nghi_dinh_168_parsed.json
